# SatQuery Phase 4 E01 — deterministic verification lanes A-C

Thin notebook: environment report, exact-repository bootstrap, one tested evaluator call, artifact verification. No scientific values are authored in this notebook.

In [ ]:
import platform
print({'python': platform.python_version(), 'platform': platform.platform()})

In [ ]:
import os, platform, subprocess, sys
from pathlib import Path
REPO_URL = os.environ.get('SATQUERY_REPO_URL', 'https://github.com/bishuk-dev/SIH-26167-SATQuery.git')
REPO_DIR = Path('/kaggle/working/SIH-26167-SATQuery')
GIT_REF = os.environ.get('SATQUERY_GIT_REF', 'HEAD')
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', '--depth=1', 'origin', GIT_REF], cwd=str(REPO_DIR), check=True)
subprocess.run(['git', 'checkout', GIT_REF], cwd=str(REPO_DIR), check=True)
head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=str(REPO_DIR), capture_output=True, text=True, check=True).stdout.strip()
assert head == GIT_REF, f'checkout mismatch: {head} != {GIT_REF}'
status = subprocess.run(['git', 'status', '--porcelain'], cwd=str(REPO_DIR), capture_output=True, text=True, check=True).stdout
assert not status.strip(), f'checkout is not clean: {status}'
print({'python': platform.python_version(), 'repo': str(REPO_DIR), 'head': head})


In [ ]:
OUTPUT_DIR = Path('/kaggle/working/satquery-output/phase4-e01-deterministic')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'ml.evaluation.run_p4_e01', '--output', str(OUTPUT_DIR / 'manifest.json')], cwd=str(REPO_DIR), check=True)

In [ ]:
manifest_path = OUTPUT_DIR / 'manifest.json'
assert manifest_path.is_file(), 'evaluator did not write manifest.json'
import json
manifest = json.loads(manifest_path.read_text())
lane_statuses = {name: lane['status'] for name, lane in manifest['lanes'].items()}
assert all(status == 'PASS' for status in lane_statuses.values()), lane_statuses
print('P4-E01 lanes:', lane_statuses)
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        print(p.relative_to(OUTPUT_DIR), p.stat().st_size)